In [3]:
import json
from urllib.request import Request, urlopen
TEAM_ID = "TEAM_32"
API_KEY = " oc__mO5aF4ZQWVd4xhI7UmY-HUWS5KYZlCf"
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"
def api(payload):
    req = Request(API_URL, data=json.dumps(payload).encode(),
    headers={"Content-Type":"application/json", "X-API-Key":API_KEY})
    with urlopen(req, timeout=30) as r:
        return json.load(r)
spec = api({"action":"spec", "team_id":TEAM_ID})
SPACE = spec["hyperparameters"]
print("Model:", spec["model"])
for name, values in SPACE.items():
    print(name, ":", values)

Model: SGDRegressor
eta0 : [0.0001, 0.001, 0.01, 0.05]
loss : ['squared_error', 'huber', 'epsilon_insensitive']
alpha : [1e-06, 3.727593720314938e-06, 1.389495494373136e-05, 5.1794746792312125e-05, 0.00019306977288832496, 0.0007196856730011514, 0.0026826957952797246, 0.01]
average : [False, True]
penalty : ['l2', 'l1', 'elasticnet']
learning_rate : ['constant', 'invscaling', 'adaptive', 'optimal']


In [ ]:
def api(payload):
    req = Request(
        API_URL,
        data=json.dumps(payload).encode(),
        headers={
            "Content-Type": "application/json",
            "X-API-Key": API_KEY
        }
    )
    with urlopen(req, timeout=30) as r:
        return json.load(r)

spec = api({
    "action": "spec",
    "team_id": TEAM_ID
})

SPACE = spec["hyperparameters"]

print("Model:", spec["model"])
for name, values in SPACE.items():
    print(name, ":", values)

call_count = 0
tested = {}

def oracle_query(params):
    global call_count

    key = tuple(params[name] for name in SPACE.keys())

    if key in tested:
        return tested[key]

    result = api({
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    })

    loss = float(result["loss"])
    tested[key] = loss
    call_count += 1

    print("Call:", call_count)
    print("Parameters:", params)
    print("Loss:", loss)
    print("-" * 60)

    return loss

best_params = {}

for name in SPACE:
    mid_index = len(SPACE[name]) // 2
    best_params[name] = SPACE[name][mid_index]

print("\nInitial parameters:")
print(best_params)

best_loss = oracle_query(best_params)

print("\nInitial Loss:", best_loss)
print()

priority_order = ["learning_rate", "loss", "eta0", "alpha", "penalty", "average"]
parameter_names = [p for p in priority_order if p in SPACE.keys()]

max_rounds = 10

for round_number in range(1, max_rounds + 1):
    print("\n")
    print("=" * 70)
    print("ROUND", round_number)
    print("=" * 70)

    improved = False

    for parameter in parameter_names:
        print("\nTesting parameter:", parameter)
        print("Current best value:", best_params[parameter])

        param_improved = False

        for value in SPACE[parameter]:
            if value == best_params[parameter]:
                continue

            trial_params = best_params.copy()
            trial_params[parameter] = value

            loss = oracle_query(trial_params)

            if loss < best_loss:
                best_loss = loss
                best_params[parameter] = value  
                
                param_improved = True
                improved = True

                print("\nNEW BEST FOUND!")
                print("Parameter:", parameter)
                print("Best value:", value)
                print("Best loss:", best_loss)

        if not param_improved:
            print("\nNo improvement for", parameter)

    if not improved:
        print("\nNo improvement in this complete round.")
        print("Optimization stopped.")
        break

print("\n")
print("=" * 70)
print("FINAL RESULT")
print("=" * 70)

print("\nBest Hyperparameters:")
for name, value in best_params.items():
    print(name, ":", value)

print("\nBest Loss:", best_loss)
print("Total Oracle Calls:", call_count)
print("Unique Configurations Tested:", len(tested))

Model: SGDRegressor
eta0 : [0.0001, 0.001, 0.01, 0.05]
loss : ['squared_error', 'huber', 'epsilon_insensitive']
alpha : [1e-06, 3.727593720314938e-06, 1.389495494373136e-05, 5.1794746792312125e-05, 0.00019306977288832496, 0.0007196856730011514, 0.0026826957952797246, 0.01]
average : [False, True]
penalty : ['l2', 'l1', 'elasticnet']
learning_rate : ['constant', 'invscaling', 'adaptive', 'optimal']

Initial parameters:
{'eta0': 0.01, 'loss': 'huber', 'alpha': 0.00019306977288832496, 'average': True, 'penalty': 'l1', 'learning_rate': 'adaptive'}
Call: 1
Parameters: {'eta0': 0.01, 'loss': 'huber', 'alpha': 0.00019306977288832496, 'average': True, 'penalty': 'l1', 'learning_rate': 'adaptive'}
Loss: 19.8886407712038
------------------------------------------------------------

Initial Loss: 19.8886407712038



ROUND 1

Testing parameter: learning_rate
Current best value: adaptive
Call: 2
Parameters: {'eta0': 0.01, 'loss': 'huber', 'alpha': 0.00019306977288832496, 'average': True, 'penalty':